# 12 — V6: Combined Features with Cross-Validation
## Customer Analytics Platform

Purpose: V5 tested one feature (review score) on a single train/test
split and got an inconclusive result - too much noise at 543 positive
examples to trust a single split. This notebook combines three features
at once (review score, delivery experience, category diversity) and
uses 5-fold cross-validation instead of one split, to get a result
that's actually trustworthy.

Leakage note: delivery_days and is_late_delivery are only knowable
once order_delivered_customer_date actually happens, which can be well
after purchase - same timing issue as review scores. Filtering only by
purchase date would leak delivery information. Delivery features here
are filtered by DELIVERY completion date, not purchase date.

In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
# load everything we need - bronze customers/reviews for clean mapping + timing,
# silver for the rest

bronze_path = "/Volumes/workspace/default/olist_raw_data/bronze"
silver_path = "/Volumes/workspace/default/olist_raw_data/silver"

customers_bronze_df = spark.read.format("delta").load(f"{bronze_path}/customers")
reviews_bronze_df    = spark.read.format("delta").load(f"{bronze_path}/reviews")

orders_df       = spark.read.format("delta").load(f"{silver_path}/orders")
payments_df     = spark.read.format("delta").load(f"{silver_path}/payments")
reviews_df      = spark.read.format("delta").load(f"{silver_path}/reviews")
order_items_df  = spark.read.format("delta").load(f"{silver_path}/order_items")
products_df     = spark.read.format("delta").load(f"{silver_path}/products")

print("all tables loaded")

In [0]:
# base join - bronze customers for clean mapping, include delivery columns this time

from pyspark.sql.functions import col

orders_payments_customers = orders_df \
    .join(payments_df, "order_id", "left") \
    .join(customers_bronze_df.select("customer_id", "customer_unique_id"), "customer_id", "left") \
    .select(
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "total_payment_value",
        "delivery_days",
        "is_late_delivery"
    ) \
    .filter(col("customer_unique_id").isNotNull())

print(f"orders joined : {orders_payments_customers.count():,}")

In [0]:
feature_cutoff_date = "2018-01-01"
label_window_end    = "2018-07-01"   # 180 days

print(f"feature window : orders before {feature_cutoff_date}")
print(f"label window   : orders from {feature_cutoff_date} to {label_window_end}")

In [0]:
# RFM - unchanged from V4/V5

from pyspark.sql.functions import (
    max as spark_max, count, sum as spark_sum, round as spark_round,
    avg as spark_avg, countDistinct, datediff, lit, to_date
)

historical_orders = orders_payments_customers.filter(
    col("order_purchase_timestamp") < feature_cutoff_date
)

rfm_historical = historical_orders \
    .groupBy("customer_unique_id") \
    .agg(
        datediff(
            to_date(lit(feature_cutoff_date)),
            spark_max("order_purchase_timestamp")
        ).alias("recency"),
        count("order_id").alias("frequency"),
        spark_round(spark_sum("total_payment_value"), 2).alias("monetary")
    ) \
    .fillna({"monetary": 0.0})

print(f"customers with history before cutoff : {rfm_historical.count():,}")

In [0]:
# review score - same logic as V5, reviews known at cutoff

reviews_combined = reviews_df.select("order_id", "review_score") \
    .join(
        reviews_bronze_df.select("order_id", "review_creation_date"),
        "order_id",
        "inner"
    )

reviews_known_at_cutoff = reviews_combined.filter(
    col("review_creation_date") < feature_cutoff_date
)

review_score_by_customer = orders_payments_customers \
    .join(reviews_known_at_cutoff.select("order_id", "review_score"), "order_id", "inner") \
    .groupBy("customer_unique_id") \
    .agg(spark_round(spark_avg("review_score"), 2).alias("avg_review_score"))

overall_avg_review = reviews_known_at_cutoff.agg(spark_avg("review_score")).collect()[0][0]

print(f"customers with known review score : {review_score_by_customer.count():,}")
print(f"overall avg review (fill value)   : {overall_avg_review:.2f}")

In [0]:
# delivery features - filtered by DELIVERY completion date, not purchase date
# an order placed before cutoff but delivered after it doesn't tell us
# delivery_days or is_late_delivery yet - that info didn't exist at the cutoff

delivered_before_cutoff = orders_payments_customers.filter(
    (col("order_purchase_timestamp") < feature_cutoff_date) &
    (col("order_delivered_customer_date").isNotNull()) &
    (col("order_delivered_customer_date") < feature_cutoff_date)
)

delivery_by_customer = delivered_before_cutoff \
    .groupBy("customer_unique_id") \
    .agg(
        spark_round(spark_avg("delivery_days"), 2).alias("avg_delivery_days"),
        spark_round(spark_avg("is_late_delivery"), 2).alias("late_delivery_rate")
    )

overall_avg_delivery = delivered_before_cutoff.agg(spark_avg("delivery_days")).collect()[0][0]
overall_late_rate    = delivered_before_cutoff.agg(spark_avg("is_late_delivery")).collect()[0][0]

print(f"customers with known delivery outcome before cutoff : {delivery_by_customer.count():,}")
print(f"overall avg delivery days (fill value) : {overall_avg_delivery:.2f}")
print(f"overall late rate (fill value)         : {overall_late_rate:.2f}")

In [0]:
# category diversity - known at purchase time, no delivery-style delay issue
# number of DISTINCT product categories a customer bought from before cutoff
# historical_orders already has customer_unique_id - no extra join needed

order_items_with_category = order_items_df \
    .join(products_df.select("product_id", "product_category_name"), "product_id", "left")

category_by_customer = historical_orders \
    .join(order_items_with_category.select("order_id", "product_category_name"), "order_id", "inner") \
    .groupBy("customer_unique_id") \
    .agg(countDistinct("product_category_name").alias("category_diversity"))

print(f"customers with category data : {category_by_customer.count():,}")
category_by_customer.show(5)

In [0]:
# future orders / target - same as V4/V5

future_orders = orders_payments_customers.filter(
    (col("order_purchase_timestamp") >= feature_cutoff_date) &
    (col("order_purchase_timestamp") < label_window_end)
)

future_activity = future_orders \
    .groupBy("customer_unique_id") \
    .agg(count("order_id").alias("future_order_count"))

print(f"customers with future activity : {future_activity.count():,}")

In [0]:
# combine ALL features - RFM + review + delivery + category

modeling_df = rfm_historical \
    .join(review_score_by_customer, "customer_unique_id", "left") \
    .join(delivery_by_customer, "customer_unique_id", "left") \
    .join(category_by_customer, "customer_unique_id", "left") \
    .join(future_activity, "customer_unique_id", "left") \
    .fillna({
        "avg_review_score": round(overall_avg_review, 2),
        "avg_delivery_days": round(overall_avg_delivery, 2),
        "late_delivery_rate": round(overall_late_rate, 2),
        "category_diversity": 0,
        "future_order_count": 0
    }) \
    .withColumn(
        "is_repeat_customer",
        (col("future_order_count") > 0).cast("int")
    )

print(f"final modeling set : {modeling_df.count():,} customers")
modeling_df.groupBy("is_repeat_customer").count().show()

In [0]:
# convert to pandas, define both feature sets

import pandas as pd

pdf = modeling_df.select(
    "recency", "frequency", "monetary",
    "avg_review_score", "avg_delivery_days", "late_delivery_rate", "category_diversity",
    "is_repeat_customer"
).toPandas()

pdf = pdf.fillna(0)

features_v4 = ["recency", "frequency", "monetary"]
features_v6 = ["recency", "frequency", "monetary",
               "avg_review_score", "avg_delivery_days", "late_delivery_rate", "category_diversity"]

X_v4 = pdf[features_v4]
X_v6 = pdf[features_v6]
y     = pdf["is_repeat_customer"]

print(f"dataset shape : {pdf.shape}")
print(f"V4 features : {features_v4}")
print(f"V6 features : {features_v6}")

In [0]:
# pull tuned params from 09 - same params for both, so it's a fair comparison
# (only the FEATURES differ between the two tests, nothing else)

import mlflow

mlflow.set_experiment(
    "/Users/elitahazelgorimanikonda@gmail.com/CLV_Customer_Segmentation"
)

runs_v3 = mlflow.search_runs(
    filter_string="tags.mlflow.runName = 'xgboost_clv_v3_tuned'",
    order_by=["metrics.roc_auc DESC"]
)

tuned_params = {
    "max_depth":        int(runs_v3.iloc[0]["params.max_depth"]),
    "learning_rate":    float(runs_v3.iloc[0]["params.learning_rate"]),
    "n_estimators":     int(runs_v3.iloc[0]["params.n_estimators"]),
    "subsample":        float(runs_v3.iloc[0]["params.subsample"]),
    "colsample_bytree": float(runs_v3.iloc[0]["params.colsample_bytree"]),
    "min_child_weight": int(runs_v3.iloc[0]["params.min_child_weight"]),
    "gamma":            float(runs_v3.iloc[0]["params.gamma"])
}

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

print(tuned_params)
print(f"scale_pos_weight : {scale_pos_weight:.2f}")

In [0]:
# 5-fold cross-validation for BOTH feature sets, same folds, same params
# this is the fair, noise-resistant comparison V5 didn't have

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = xgb.XGBClassifier(
    **tuned_params,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    verbosity=0
)

scores_v4 = cross_val_score(model, X_v4, y, cv=cv, scoring="roc_auc", n_jobs=-1)
scores_v6 = cross_val_score(model, X_v6, y, cv=cv, scoring="roc_auc", n_jobs=-1)

print("V4 (RFM only) - 5-fold ROC AUC:")
print(f"  scores : {[round(s, 4) for s in scores_v4]}")
print(f"  mean   : {scores_v4.mean():.4f}  (+/- {scores_v4.std():.4f})")
print()
print("V6 (RFM + review + delivery + category) - 5-fold ROC AUC:")
print(f"  scores : {[round(s, 4) for s in scores_v6]}")
print(f"  mean   : {scores_v6.mean():.4f}  (+/- {scores_v6.std():.4f})")

In [0]:
# log the cross-validation comparison to mlflow, then fit V6 on full data
# for the deployable model artifact

import xgboost as xgb
from sklearn.metrics import roc_auc_score

with mlflow.start_run(run_name="xgboost_clv_v6_combined_features"):

    mlflow.log_param("features", ", ".join(features_v6))
    mlflow.log_param("cv_folds", 5)
    mlflow.log_params(tuned_params)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)

    mlflow.log_metric("cv_roc_auc_mean_v4_baseline", scores_v4.mean())
    mlflow.log_metric("cv_roc_auc_std_v4_baseline", scores_v4.std())
    mlflow.log_metric("cv_roc_auc_mean_v6", scores_v6.mean())
    mlflow.log_metric("cv_roc_auc_std_v6", scores_v6.std())
    mlflow.log_metric("folds_won_v6_of_5", sum(scores_v6 > scores_v4))

    # fit final model on all data for deployment
    final_model_v6 = xgb.XGBClassifier(
        **tuned_params,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    final_model_v6.fit(X_v6, y)

    mlflow.xgboost.log_model(final_model_v6, "xgboost_clv_v6_combined")

    print("V6 logged to mlflow")
    print(f"V4 CV mean : {scores_v4.mean():.4f}")
    print(f"V6 CV mean : {scores_v6.mean():.4f}")
    print(f"V6 won {sum(scores_v6 > scores_v4)} of 5 folds")

In [0]:
# re-log v6 with a signature and input example - needed for Model Serving
# to know the expected input schema (recency, frequency, monetary, etc.)

with mlflow.start_run(run_name="xgboost_clv_v6_serving_ready"):
    mlflow.log_params(tuned_params)
    mlflow.log_param("features", ", ".join(features_v6))

    mlflow.xgboost.log_model(
        final_model_v6,
        "model",
        input_example=X_v6.iloc[:5]
    )

    print("V6 re-logged with signature for serving")

In [0]:
# register the model to Unity Catalog - required for it to appear
# in Model Serving's "My models" list

model_uri = f"runs:/{mlflow.last_active_run().info.run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="workspace.default.clv_v6_repeat_purchase"
)

print(f"registered as: {registered_model.name}")
print(f"version: {registered_model.version}")

In [0]:
# wrap the model so serving returns probability, not hard 0/1 labels
# consistent with how 08_Model_Deployment uses predict_proba for CLV tiers

import mlflow.pyfunc

class ProbabilityWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import xgboost as xgb
        self.model = xgb.XGBClassifier()
        self.model.load_model(context.artifacts["xgb_model"])

    def predict(self, context, model_input):
        # return probability of class 1 (repeat purchase), not the hard label
        return self.model.predict_proba(model_input)[:, 1]

# save the raw xgboost model to a temp path so the wrapper can load it
final_model_v6.save_model("/tmp/v6_xgb_model.json")

with mlflow.start_run(run_name="xgboost_clv_v6_probability_serving"):
    mlflow.log_params(tuned_params)
    mlflow.log_param("features", ", ".join(features_v6))
    mlflow.log_param("output", "probability_of_repeat_purchase")

    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ProbabilityWrapper(),
        artifacts={"xgb_model": "/tmp/v6_xgb_model.json"},
        input_example=X_v6.iloc[:5]
    )

    run_id_for_serving = mlflow.active_run().info.run_id
    print(f"probability-output model logged, run_id: {run_id_for_serving}")

In [0]:
model_uri = f"runs:/{run_id_for_serving}/model"

registered_model_v2 = mlflow.register_model(
    model_uri=model_uri,
    name="workspace.default.clv_v6_repeat_purchase"
)

print(f"registered as version: {registered_model_v2.version}")

## V6 Summary — Combined Features with Cross-Validation

Tested whether combining review score, delivery experience, and product
category diversity improves on V4's RFM-only baseline. Unlike V5 (single
train/test split), this used 5-fold stratified cross-validation for a
statistically defensible comparison.

| Model | Features | CV ROC AUC (mean ± std) |
|-------|----------|--------------------------|
| V4 baseline | recency, frequency, monetary | 0.4995 ± 0.0218 |
| V6 | + avg_review_score, avg_delivery_days, late_delivery_rate, category_diversity | 0.5232 ± 0.0312 |

**Result: V6 outperformed V4 in all 5 of 5 folds** - under a fair paired
comparison (same folds, same customers, same hyperparameters), this
has roughly a 3% probability of happening by chance alone (1/2^5) if
the features made no real difference. This is genuine, if modest,
evidence that richer features help - not noise.

Note: V4's cross-validated mean (0.4995) is meaningfully lower than the
single-split result reported in 10_Temporal_CLV_Target (0.5461) - this
is expected and important. It shows that single train/test splits at
this sample size (543 positive examples) can be misleadingly optimistic
or pessimistic. Cross-validation is the more trustworthy comparison
method, and by that standard, RFM alone is statistically indistinguishable
from random guessing (0.4995 ≈ 0.50), while the combined feature set
shows a small, real improvement (0.5232).

Target renamed to `is_repeat_customer` (dropped the misleading `_90d`
suffix carried over from earlier notebooks where the window was
actually variable, not fixed at 90 days).

Next Step:
- V6 is now the strongest honestly-measured model - this is the one
  that should back real-time serving and drift monitoring going forward
- Further gains would likely need genuinely new signal (seasonality,
  customer service contact history, browsing behavior) rather than
  more tuning on the current feature set
- Model logged to MLflow as xgboost_clv_v6_combined_features